In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
import glob
import sys
import copy
import random
import matplotlib.pyplot as plt
import matplotlib
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from xgboost import XGBRegressor as XGBR

## 100 Random divisions of XGB

In [9]:
data_path = r"path\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")

data_features = data.iloc[:, :10]
data_label = data.iloc[:, -1]

X_all = data_features.to_numpy(dtype=np.float32)
y_all = data_label.to_numpy(dtype=np.float32).ravel()

print("All data shape:", data.shape)
print("Feature shape:", X_all.shape)
print("Label shape:", y_all.shape)


N_SPLITS = 100
TEST_SIZE = 0.3

XGB_N_ESTIMATORS = 50
XGB_MAX_DEPTH = 6

def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    r2 = r2_score(y_eval, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
    return r2, rmse, y_pred

All data shape: (102172, 11)
Feature shape: (102172, 10)
Label shape: (102172,)


In [ ]:
data_features = data.iloc[:, :10]
data_label = data.iloc[:, -1]

X_all = data_features.to_numpy(dtype=np.float32)
y_all = data_label.to_numpy(dtype=np.float32).ravel()

print("All data shape:", data.shape)
print("Feature shape:", X_all.shape)
print("Label shape:", y_all.shape)

N_EXPERIMENTS = 178
POINTS_PER_EXPERIMENT = 574

expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)
print("Actual rows / 574:", actual_rows / POINTS_PER_EXPERIMENT)


experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

print("\nExperiment_ID count:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().head())
print(data_with_id["Experiment_ID"].value_counts().sort_index().tail())


N_SPLITS = 100
TEST_SIZE = 0.3

XGB_N_ESTIMATORS = 50
XGB_MAX_DEPTH = 6


def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    r2 = r2_score(y_eval, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
    return r2, rmse, y_pred


results = []

all_experiment_ids = np.arange(N_EXPERIMENTS)

for i in range(N_SPLITS):
    random_seed = 100 + i

    train_exp_ids, test_exp_ids = train_test_split(
        all_experiment_ids,
        test_size=TEST_SIZE,
        random_state=random_seed,
        shuffle=True
    )

    train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
    test_mask = data_with_id["Experiment_ID"].isin(test_exp_ids).to_numpy()

    feature_train = X_all[train_mask]
    label_train = y_all[train_mask]

    feature_test = X_all[test_mask]
    label_test = y_all[test_mask]

    model = XGBRegressor(
        n_estimators=XGB_N_ESTIMATORS,
        max_depth=XGB_MAX_DEPTH,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(feature_train, label_train)

    train_r2, train_rmse, _ = evaluate_model(
        model,
        feature_train,
        label_train
    )

    test_r2, test_rmse, _ = evaluate_model(
        model,
        feature_test,
        label_test
    )

    results.append({
        "Split": i + 1,
        "Split_random_state": random_seed,

        "Train_experiment_number": len(train_exp_ids),
        "Test_experiment_number": len(test_exp_ids),

        "Train_data_points": len(label_train),
        "Test_data_points": len(label_test),

        "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
        "Test_experiment_IDs": ",".join(map(str, sorted(test_exp_ids))),

        "n_estimators": XGB_N_ESTIMATORS,
        "max_depth": XGB_MAX_DEPTH,

        "Train_R2": train_r2,
        "Test_R2": test_r2,
        "Train_RMSE": train_rmse,
        "Test_RMSE": test_rmse,

        "Delta_R2_Train_minus_Test": train_r2 - test_r2,
        "Delta_RMSE_Test_minus_Train": test_rmse - train_rmse
    })

    print(
        f"Split {i+1:03d}/{N_SPLITS} | "
        f"Train_exp={len(train_exp_ids)}, Test_exp={len(test_exp_ids)} | "
        f"Train_points={len(label_train)}, Test_points={len(label_test)} | "
        f"Train_R2={train_r2:.6f}, Test_R2={test_r2:.6f}, "
        f"Train_RMSE={train_rmse:.6f}, Test_RMSE={test_rmse:.6f}"
    )


## 100 Random divisions of DNN

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [12]:
data_path = r"path\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")

data_features = data.iloc[:, :10]
data_label = data.iloc[:, -1]

X_all = data_features.to_numpy(dtype=np.float32)
y_all = data_label.to_numpy(dtype=np.float32).reshape(-1, 1)

print("All data shape:", data.shape)
print("Feature shape:", X_all.shape)
print("Label shape:", y_all.shape)

All data shape: (102172, 11)
Feature shape: (102172, 10)
Label shape: (102172, 1)


In [16]:
expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)

N_EXPERIMENTS = 178
POINTS_PER_EXPERIMENT = 574
experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

print("\nExperiment_ID count:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().head())
print(data_with_id["Experiment_ID"].value_counts().sort_index().tail())


N_SPLITS = 100
TEST_SIZE = 0.3

n_epochs = 50

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class ExcelDataset(Dataset):
    def __init__(self, feature_train, label_train):
        self.feature = feature_train
        self.label = label_train
        self.x = torch.from_numpy(self.feature).float()
        self.y = torch.from_numpy(self.label).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        x = self.x[index]
        y = self.y[index]
        return {"x": x, "y": y}


class BP(nn.Module):
    def __init__(self, num_layers, hidden_size):
        super(BP, self).__init__()
        self.Liner1 = nn.Linear(10, hidden_size)
        self.queue = [
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.LeakyReLU(0.1)
            )
            for _ in range(num_layers - 1)
        ]
        self.model = nn.Sequential(*self.queue)
        self.Liner2 = nn.Linear(hidden_size, 1)

    def forward(self, input):
        output = self.Liner1(input)
        output = self.model(output)
        output = self.Liner2(output)
        return output



def evaluate_dnn(bp, feature_eval, label_eval):
    bp.eval()

    x_eval = torch.from_numpy(feature_eval).float()

    with torch.no_grad():
        pred = bp(x_eval).detach().cpu().numpy().reshape(-1)

    true = label_eval.reshape(-1)

    r2 = r2_score(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))

    return r2, rmse, pred


results = []

all_experiment_ids = np.arange(N_EXPERIMENTS)

for i in range(N_SPLITS):
    split_random_seed = 100 + i

    train_exp_ids, test_exp_ids = train_test_split(
        all_experiment_ids,
        test_size=TEST_SIZE,
        random_state=split_random_seed,
        shuffle=True
    )

    train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
    test_mask = data_with_id["Experiment_ID"].isin(test_exp_ids).to_numpy()

    feature_train = X_all[train_mask]
    label_train = y_all[train_mask]

    feature_test = X_all[test_mask]
    label_test = y_all[test_mask]

    
    set_seed(split_random_seed)

    dataset = ExcelDataset(feature_train, label_train)

    dataloader = DataLoader(
        dataset=dataset,
        batch_size=256,
        shuffle=True,
        num_workers=0,
        drop_last=True
    )

    bp = BP(3, 12)

    optimizer = torch.optim.Adam(
        bp.parameters(),
        lr=0.01
    )

    MSEloss = nn.MSELoss()

    epoch_loss_history = []

    for epoch in range(n_epochs):
        bp.train()
        batch_loss_list = []

        for batch in dataloader:
            x = batch["x"]
            y = batch["y"]

            optimizer.zero_grad()

            pred = bp(x)
            loss = MSEloss(pred, y)

            loss.backward()
            optimizer.step()

            batch_loss_list.append(loss.item())

        epoch_loss = np.mean(batch_loss_list)
        epoch_loss_history.append(epoch_loss)


    train_r2, train_rmse, _ = evaluate_dnn(
        bp,
        feature_train,
        label_train
    )

    test_r2, test_rmse, _ = evaluate_dnn(
        bp,
        feature_test,
        label_test
    )

    results.append({
        "Split": i + 1,
        "Split_random_state": split_random_seed,

        "Train_experiment_number": len(train_exp_ids),
        "Test_experiment_number": len(test_exp_ids),

        "Train_data_points": len(label_train),
        "Test_data_points": len(label_test),

        "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
        "Test_experiment_IDs": ",".join(map(str, sorted(test_exp_ids))),

        "num_layers": 3,
        "hidden_size": 12,
        "n_epochs": n_epochs,
        "batch_size": 256,
        "learning_rate": 0.01,
        "drop_last": True,

        "Final_train_epoch_loss": epoch_loss_history[-1],

        "Train_R2": train_r2,
        "Test_R2": test_r2,
        "Train_RMSE": train_rmse,
        "Test_RMSE": test_rmse,

        "Delta_R2_Train_minus_Test": train_r2 - test_r2,
        "Delta_RMSE_Test_minus_Train": test_rmse - train_rmse
    })

    print(
        f"Split {i+1:03d}/{N_SPLITS} | "
        f"seed={split_random_seed} | "
        f"Train_exp={len(train_exp_ids)}, Test_exp={len(test_exp_ids)} | "
        f"Train_R2={train_r2:.6f}, Test_R2={test_r2:.6f}, "
        f"Train_RMSE={train_rmse:.6f}, Test_RMSE={test_rmse:.6f}"
    )

Expected rows: 102172
Actual rows: 102172

Experiment_ID count:
Experiment_ID
0    574
1    574
2    574
3    574
4    574
Name: count, dtype: int64
Experiment_ID
173    574
174    574
175    574
176    574
177    574
Name: count, dtype: int64
Split 001/100 | seed=100 | Train_exp=124, Test_exp=54 | Train_R2=0.988085, Test_R2=0.976664, Train_RMSE=3.897326, Test_RMSE=5.506373


KeyboardInterrupt: 

## 100 Random divisions of MQR

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


data_path = r"path\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")


single_bc_path = r"path\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")
single_bc_data = pd.read_excel(data_path,sheet_name="1BC")

print("All data shape:", data.shape)
print("All data columns:")
print(data.columns.tolist())

print("\nSingle BC data shape:", single_bc_data.shape)
print("Single BC data columns:")
print(single_bc_data.columns.tolist())
N_EXPERIMENTS = 178
POINTS_PER_EXPERIMENT = 574


bc_cols = [
    "Cellulose",
    "Hemicellulose",
    "Lignin",
    "PE",
    "PS",
    "PP",
    "PET",
    "PVC",
    "Starch"
]

time_col = "Time"
label_col = "TG"

bc_name_map = {
    "Cellulose": "Cellulose",
    "Hemicellulose": "Hemicellulose",
    "Lignin": "Lignin",
    "PE": "PE",
    "PS": "PS",
    "PP": "PP",
    "PET": "PET",
    "PVC": "PVC",
    "Starch": "Starch"
}


expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("\nExpected rows:", expected_rows)
print("Actual rows:", actual_rows)
print("Actual rows / 574:", actual_rows / POINTS_PER_EXPERIMENT)



experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

print("\nExperiment_ID count head:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().head())

print("\nExperiment_ID count tail:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().tail())

All data shape: (102172, 11)
All data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG']

Single BC data shape: (5166, 11)
Single BC data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG(wt.%)']

Expected rows: 102172
Actual rows: 102172
Actual rows / 574: 178.0

Experiment_ID count head:
Experiment_ID
0    574
1    574
2    574
3    574
4    574
Name: count, dtype: int64

Experiment_ID count tail:
Experiment_ID
173    574
174    574
175    574
176    574
177    574
Name: count, dtype: int64


In [12]:
def build_single_bc_tg_matrix(single_bc_data, bc_cols, bc_name_map):
    single_tg_dict = {}

    for bc in bc_cols:
        single_col = bc_name_map[bc]
        one_bc_curve = single_bc_data.loc[
            single_bc_data[single_col] == 1,
            ["Time", "TG(wt.%)"]
        ].copy()

        one_bc_curve = one_bc_curve.reset_index(drop=True)

        single_tg_dict[bc] = one_bc_curve["TG(wt.%)"].to_numpy(dtype=np.float32)

    first_bc = bc_cols[0]
    first_single_col = bc_name_map[first_bc]

    time_array = single_bc_data.loc[
        single_bc_data[first_single_col] == 1,
        "Time"
    ].reset_index(drop=True).to_numpy(dtype=np.float32)

    single_tg_df = pd.DataFrame(single_tg_dict)
    single_tg_df.insert(0, time_col, time_array)

    return single_tg_df


single_tg_df = build_single_bc_tg_matrix(
    single_bc_data=single_bc_data,
    bc_cols=bc_cols,
    bc_name_map=bc_name_map
)

In [13]:
def build_mqr_features_old_fusion(
    df,
    single_tg_df,
    bc_cols,
    time_col,
    use_percent_composition=True
):
    df = df.copy().reset_index(drop=True)

    bcs_values = df[bc_cols].to_numpy(dtype=np.float32)

    #
    max_bcs = np.nanmax(bcs_values)

    if use_percent_composition and max_bcs <= 1.5:
        bcs_values = bcs_values * 100.0

    data_time = df[time_col].to_numpy(dtype=np.float32)
    single_time = single_tg_df[time_col].to_numpy(dtype=np.float32)

    time_indices = np.array([
        np.argmin(np.abs(single_time - t))
        for t in data_time
    ])

    max_time_diff = np.max(np.abs(data_time - single_time[time_indices]))


    single_tg_values = single_tg_df[bc_cols].to_numpy(dtype=np.float32)
    single_tg_for_each_row = single_tg_values[time_indices, :]

    #Features mxiture
    fused_features = bcs_values * single_tg_for_each_row / 100.0

    fused_df = pd.DataFrame(
        fused_features,
        columns=bc_cols,
        index=df.index
    )

    return fused_df



data_features = build_mqr_features_old_fusion(
    df=data_with_id,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    use_percent_composition=True
)

data_label = data_with_id[label_col].to_numpy(dtype=np.float32).ravel()

In [14]:
def train_and_evaluate_mqr(X_train, X_test, y_train, y_test):
    poly = PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )

    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    X_train_poly[:, 9:] = X_train_poly[:, 9:] / 100.0
    X_test_poly[:, 9:] = X_test_poly[:, 9:] / 100.0

    feature_names = poly.get_feature_names_out(X_train.columns)

    to_keep = [
        idx for idx, name in enumerate(feature_names)
    ]

    X_train_poly_reduced = X_train_poly[:, to_keep]
    X_test_poly_reduced = X_test_poly[:, to_keep]

    reduced_feature_names = [feature_names[idx] for idx in to_keep]

    model = LinearRegression()
    model.fit(X_train_poly_reduced, y_train)

    y_train_pred = model.predict(X_train_poly_reduced)
    y_test_pred = model.predict(X_test_poly_reduced)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)

    coefficients = model.coef_
    intercept = model.intercept_

    coef_df = pd.DataFrame({
        "Feature": reduced_feature_names,
        "Coefficient": coefficients
    })

    return {
        "model": model,
        "poly": poly,
        "to_keep": to_keep,
        "reduced_feature_names": reduced_feature_names,
        "coef_df": coef_df,
        "intercept": intercept,
        "Train_R2": train_r2,
        "Test_R2": test_r2,
        "Train_MSE": train_mse,
        "Test_MSE": test_mse,
        "Train_RMSE": train_rmse,
        "Test_RMSE": test_rmse
    }

In [15]:
random_int = random.sample(range(1, 1001), 100)

score_R2 = np.zeros(100)
score_MSE = np.zeros(100)
score_RMSE = np.zeros(100)

score_train_R2 = np.zeros(100)
score_train_MSE = np.zeros(100)
score_train_RMSE = np.zeros(100)

results = []

all_experiment_ids = np.arange(N_EXPERIMENTS)


for i in range(100):
    random_seed = random_int[i]

    train_exp_ids, test_exp_ids = train_test_split(
        all_experiment_ids,
        test_size=0.3,
        random_state=random_seed,
        shuffle=True
    )

    train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
    test_mask = data_with_id["Experiment_ID"].isin(test_exp_ids).to_numpy()

    X_train = data_features.loc[train_mask, :].copy()
    X_test = data_features.loc[test_mask, :].copy()

    y_train = data_label[train_mask]
    y_test = data_label[test_mask]

    eval_result = train_and_evaluate_mqr(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )

    score_R2[i] = eval_result["Test_R2"]
    score_MSE[i] = eval_result["Test_MSE"]
    score_RMSE[i] = eval_result["Test_RMSE"]

    score_train_R2[i] = eval_result["Train_R2"]
    score_train_MSE[i] = eval_result["Train_MSE"]
    score_train_RMSE[i] = eval_result["Train_RMSE"]

    results.append({
        "Split": i + 1,
        "Random_state": random_seed,

        "Train_experiment_number": len(train_exp_ids),
        "Test_experiment_number": len(test_exp_ids),

        "Train_data_points": len(y_train),
        "Test_data_points": len(y_test),

        "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
        "Test_experiment_IDs": ",".join(map(str, sorted(test_exp_ids))),

        "Train_R2": eval_result["Train_R2"],
        "Test_R2": eval_result["Test_R2"],

        "Train_MSE": eval_result["Train_MSE"],
        "Test_MSE": eval_result["Test_MSE"],

        "Train_RMSE": eval_result["Train_RMSE"],
        "Test_RMSE": eval_result["Test_RMSE"],

        "Intercept": eval_result["intercept"]
    })

    print(
        f"Split {i+1:03d}/100 | "
        f"random_state={random_seed} | "
        f"Train_R2={eval_result['Train_R2']:.6f}, "
        f"Test_R2={eval_result['Test_R2']:.6f}, "
        f"Train_RMSE={eval_result['Train_RMSE']:.6f}, "
        f"Test_RMSE={eval_result['Test_RMSE']:.6f}"
    )



score_R2_df = pd.DataFrame(score_R2, columns=["Test_R2"])
score_MSE_df = pd.DataFrame(score_MSE, columns=["Test_MSE"])
score_RMSE_df = pd.DataFrame(score_RMSE, columns=["Test_RMSE"])

score_train_R2_df = pd.DataFrame(score_train_R2, columns=["Train_R2"])
score_train_MSE_df = pd.DataFrame(score_train_MSE, columns=["Train_MSE"])
score_train_RMSE_df = pd.DataFrame(score_train_RMSE, columns=["Train_RMSE"])

score_summary_df = pd.DataFrame({
    "Split": np.arange(1, 101),
    "Random_state": random_int,

    "Train_R2": score_train_R2,
    "Test_R2": score_R2,

    "Train_MSE": score_train_MSE,
    "Test_MSE": score_MSE,

    "Train_RMSE": score_train_RMSE,
    "Test_RMSE": score_RMSE
})

Split 001/100 | random_state=73 | Train_R2=0.953482, Test_R2=0.950624, Train_RMSE=7.834958, Test_RMSE=7.678123
Split 002/100 | random_state=369 | Train_R2=0.955761, Test_R2=0.607343, Train_RMSE=7.466517, Test_RMSE=22.878288
Split 003/100 | random_state=887 | Train_R2=0.959449, Test_R2=-8.472739, Train_RMSE=7.229759, Test_RMSE=109.548241
Split 004/100 | random_state=653 | Train_R2=0.957548, Test_R2=-0.257740, Train_RMSE=7.428287, Test_RMSE=39.503109
Split 005/100 | random_state=716 | Train_R2=0.955669, Test_R2=-0.540963, Train_RMSE=7.579550, Test_RMSE=43.890930
Split 006/100 | random_state=129 | Train_R2=0.958333, Test_R2=0.214548, Train_RMSE=7.214502, Test_RMSE=32.634422
Split 007/100 | random_state=908 | Train_R2=0.956991, Test_R2=-2.390547, Train_RMSE=7.427234, Test_RMSE=65.915421
Split 008/100 | random_state=702 | Train_R2=0.952091, Test_R2=-0.743356, Train_RMSE=7.885098, Test_RMSE=46.613056
Split 009/100 | random_state=108 | Train_R2=0.955770, Test_R2=-2.676088, Train_RMSE=7.560351

KeyboardInterrupt: 